In [0]:
# =============================================================
# The "Salt Method" in PySpark — fixing data skew
# =============================================================
# Problem: When one key (or a few keys) has vastly more rows than
# others, ALL rows for that key get hashed to the SAME partition,
# so one task does almost all the work while others sit idle.
#
# Fix: "Salt" the skewed key — append a random value (0..N-1) to
# spread one hot key across N partitions:
#   - Skewed side:   key -> key + '_' + salt(rand 0..N-1)
#   - Other side:    replicate each row N times (key + '_0' ... key + '_N-1')
#     so every salted key still finds its join partner.
# =============================================================

import pyspark.sql.functions as F

N_SALTS = 8  # number of salt buckets / parallel tasks per hot key

# --- 1. Create skewed demo data --------------------------------
# 90% of rows share hot key "a"
skewed_df = (
    spark.range(1_000_000)
    .withColumn("key", F.when(F.col("id") % 10 < 9, F.lit("a"))
                        .otherwise(F.concat(F.lit("k_"), (F.col("id") % 10).cast("string"))))
    .withColumn("value", F.col("id") % 100)
)

small_df = spark.createDataFrame(
    [("a", "hot"), ("k_0", "c0"), ("k_1", "c1"), ("k_2", "c2"),
     ("k_3", "c3"), ("k_4", "c4"), ("k_5", "c5"), ("k_6", "c6"),
     ("k_7", "c7"), ("k_8", "c8"), ("k_9", "c9")],
    ["key", "label"],
)

# --- 2. Baseline: naive join (skewed, 1 giant partition) ---------
naive = skewed_df.join(small_df, "key")
print("Naive join row count:", naive.count())

# --- 3. Salted join ---------------------------------------------
# Skewed side: append a random salt to the hot key
skewed_salted = skewed_df.withColumn(
    "salted_key",
    F.concat(F.col("key"), F.lit("_"), (F.rand() * N_SALTS).cast("int").cast("string")),
)

# Small side: explode each row into N_SALTS copies, one per salt value,
# so every salted key on the left still has a match on the right
salts = F.explode(F.array([F.lit(i) for i in range(N_SALTS)]))
small_salted = (
    small_df
    .withColumn("salt", salts)
    .withColumn("salted_key", F.concat(F.col("key"), F.lit("_"), F.col("salt").cast("string")))
    .drop("salt")
)

salted_join = skewed_salted.join(small_salted, "salted_key")
print("Salted join row count:", salted_join.count())  # same result, better balance

# --- 4. Salting for skewed aggregations -------------------------
# A GROUP BY on the hot key also funnels into one task.
# Salt -> aggregate per bucket -> merge the buckets:
salted_agg = (
    skewed_df
    .withColumn("salt", (F.rand() * N_SALTS).cast("int"))
    .groupBy("key", "salt")                 # balanced partial aggregation
    .agg(F.sum("value").alias("partial_sum"), F.count("*").alias("partial_cnt"))
    .groupBy("key")                          # final merge over N small buckets
    .agg(F.sum("partial_sum").alias("total"), F.sum("partial_cnt").alias("cnt"))
)
salted_agg.orderBy(F.desc("cnt")).show()

# --- 5. Key takeaways --------------------------------------------
# - Spark 3.x AQE skew join handling (spark.sql.adaptive.enabled=true and
#   spark.sql.adaptive.skewJoin.enabled=true, both default on) fixes many
#   skew cases automatically — try that before manual salting.
# - Salting is worth it when AQE can't help (huge single keys, skewed
#   aggregations, or older runtimes).
# - Only salt keys you KNOW are hot (from the Spark UI); salting everything
#   adds shuffle volume and replication overhead.
# - Tune N_SALTS roughly to the number of cores; more salts = more
#   parallelism but more rows on the replicated side.

Naive join row count: 1000000
Salted join row count: 1000000
+---+--------+------+
|key|   total|   cnt|
+---+--------+------+
|  a|44100000|900000|
|k_9| 5400000|100000|
+---+--------+------+



In [0]:
# =============================================================
# 2. Prove the salted results are CORRECT, not just balanced
# =============================================================
# Salting rewrites the join key, so always check the salted output
# against the naive output before trusting it in production.

naive_count = naive.count()
salted_count = salted_join.count()
print(f"naive join rows : {naive_count}")
print(f"salted join rows: {salted_count}")
assert naive_count == salted_count, "row counts differ!"

# Per-key aggregates must match too
naive_by_key = naive.groupBy("key").agg(
    F.sum("value").alias("s_naive"), F.count("*").alias("c_naive")
)
# NOTE: salted_join has TWO 'key' columns (both sides kept theirs), so
# grouping on "key" is ambiguous. Instead, recover the original key by
# stripping the "_<salt>" suffix from salted_key.
salted_by_key = (
    salted_join
    .withColumn("orig_key", F.regexp_replace("salted_key", r"_\d+$", ""))
    .groupBy("orig_key")
    .agg(F.sum("value").alias("s_salted"), F.count("*").alias("c_salted"))
    .withColumnRenamed("orig_key", "key")
)

mismatched = (
    naive_by_key.join(salted_by_key, "key")
    .filter((F.col("s_naive") != F.col("s_salted")) |
            (F.col("c_naive") != F.col("c_salted")))
    .count()
)
print(f"join keys with mismatched aggregates: {mismatched}")
assert mismatched == 0

# Same check for the salted AGGREGATION (cell 1, section 4):
plain_agg = skewed_df.groupBy("key").agg(
    F.sum("value").alias("plain_total"), F.count("*").alias("plain_cnt")
)
agg_mismatch = (
    salted_agg.join(plain_agg, "key")
    .filter((F.col("total") != F.col("plain_total")) |
            (F.col("cnt") != F.col("plain_cnt")))
    .count()
)
print(f"aggregation keys with wrong totals: {agg_mismatch}")
assert agg_mismatch == 0
print("ALL CHECKS PASSED")

naive join rows : 1000000
salted join rows: 1000000
join keys with mismatched aggregates: 0
aggregation keys with wrong totals: 0
ALL CHECKS PASSED


## 3. Measure the balance: before vs after salting

On serverless compute we can't inspect raw shuffle partitions, but **rows per key** is a good proxy: before salting, the hot key `a` is one giant partition (~90% of all rows); after salting, it spreads across `N_SALTS` buckets of roughly equal size, each of which becomes an independent join task.

In [0]:
# =============================================================
# 3. Measure the balance: before vs after salting
# =============================================================
print("BEFORE salting -- one key dominates (this is the giant task):")
skewed_df.groupBy("key").count().orderBy(F.desc("count")).show(5, truncate=False)

print(f"AFTER salting -- the hot key is spread over {N_SALTS} buckets:")
(skewed_salted.groupBy("salted_key").count()
 .where(F.col("salted_key").startswith("a_"))
 .orderBy("salted_key")
 .show(N_SALTS, truncate=False))

BEFORE salting -- one key dominates (this is the giant task):
+---+------+
|key|count |
+---+------+
|a  |900000|
|k_9|100000|
+---+------+

AFTER salting -- the hot key is spread over 8 buckets:
+----------+------+
|salted_key|count |
+----------+------+
|a_0       |112114|
|a_1       |112126|
|a_2       |112520|
|a_3       |113403|
|a_4       |112518|
|a_5       |112092|
|a_6       |112455|
|a_7       |112772|
+----------+------+



## 4. Pitfall: `rand()` salts are not reproducible

`F.rand()` is re-evaluated every time Spark recomputes a row — on task retries, or whenever a stage re-runs before its shuffle output is reused. That can silently change a row's salt mid-job, which breaks any logic that assumes the salt is stable (re-joins, dedup, debugging).

A **deterministic** salt derives the bucket from a stable column with `pmod(hash(<col>), N)`, so the same row always lands in the same bucket.

In [0]:
# =============================================================
# 4. Deterministic salting: pmod(hash(id), N) instead of rand()
# =============================================================
skewed_hash_salted = skewed_df.withColumn(
    "salted_key",
    F.concat(F.col("key"), F.lit("_"),
             F.pmod(F.hash(F.col("id")), F.lit(N_SALTS)).cast("string")),
)

# Same replication trick on the small side (select + explode works on every
# Spark version, unlike withColumn with a generator on older releases)
small_hash_salted = (
    small_df
    .select(F.col("key"), F.col("label"),
            F.explode(F.array([F.lit(i) for i in range(N_SALTS)])).alias("salt"))
    .withColumn("salted_key",
                F.concat(F.col("key"), F.lit("_"), F.col("salt").cast("string")))
    .drop("salt")
)

hash_salted_join = skewed_hash_salted.join(small_hash_salted, "salted_key")
hash_count = hash_salted_join.count()
print(f"hash-salted join rows: {hash_count} (naive: {naive_count})")
assert hash_count == naive_count

# The hash salt balances the hot key just as well as rand()
(skewed_hash_salted.groupBy("salted_key").count()
 .where(F.col("salted_key").startswith("a_"))
 .orderBy("salted_key")
 .show(N_SALTS, truncate=False))

hash-salted join rows: 1000000 (naive: 1000000)
+----------+------+
|salted_key|count |
+----------+------+
|a_0       |112219|
|a_1       |112501|
|a_2       |112521|
|a_3       |112543|
|a_4       |112684|
|a_5       |112627|
|a_6       |112632|
|a_7       |112273|
+----------+------+



## 5. Salt ONLY the keys you know are hot

Salting every key replicates the small side `N_SALTS` times for keys that were perfectly balanced to begin with. In production, salt **only known hot keys** (discovered from the Spark UI or a per-key count) and leave cold keys untouched — the small side then needs replication only for the hot keys, keeping shuffle volume and join fan-out low.

In [0]:
# =============================================================
# 5. Targeted salting: salt ONLY the known hot keys
# =============================================================
HOT_KEYS = ["a"]  # in practice, discovered from the Spark UI / per-key counts

# Skewed side: hot keys get a deterministic salt, cold keys keep their key
skewed_targeted = skewed_df.withColumn(
    "salted_key",
    F.when(F.col("key").isin(HOT_KEYS),
           F.concat(F.col("key"), F.lit("_"),
                    F.pmod(F.hash(F.col("id")), F.lit(N_SALTS)).cast("string")))
     .otherwise(F.col("key")),
)

# Small side: replicate ONLY the hot-key rows; cold-key rows stay single
hot_side = (
    small_df.filter(F.col("key").isin(HOT_KEYS))
    .select(F.col("key"), F.col("label"),
            F.explode(F.array([F.lit(i) for i in range(N_SALTS)])).alias("salt"))
    .withColumn("salted_key",
                F.concat(F.col("key"), F.lit("_"), F.col("salt").cast("string")))
    .drop("key", "salt")
)
cold_side = (
    small_df.filter(~F.col("key").isin(HOT_KEYS))
    .withColumnRenamed("key", "salted_key")
)
small_targeted = hot_side.unionByName(cold_side)

targeted_join = skewed_targeted.join(small_targeted, "salted_key")
targeted_count = targeted_join.count()
print(f"targeted-salt join rows: {targeted_count} (naive: {naive_count})")
assert targeted_count == naive_count

# Replication cost comparison: full salting replicates EVERY small row N
# times; targeted salting replicates only the hot ones
print(f"small side rows -- full salting: {small_salted.count()}, "
      f"targeted: {small_targeted.count()} (original: {small_df.count()})")

targeted-salt join rows: 1000000 (naive: 1000000)
small side rows -- full salting: 88, targeted: 18 (original: 11)


## Decision guide: when to salt

| Situation | Do this |
| --- | --- |
| Skewed join on a recent Databricks runtime | Try AQE first — skew-join splitting is on by default and handles most cases |
| One key so huge that even AQE's runtime splits are still slow, or you need guaranteed balance | Manual salting (cells above) |
| Skewed `GROUP BY` / aggregation | AQE doesn't split aggregations — salt + two-stage aggregation (cell 1, section 4) |
| Only a few keys are hot | **Targeted salting** — salt only those keys, avoid N× replication of the whole small side |

Rules of thumb:

* Discover hot keys from evidence (Spark UI task skew, or `groupBy(key).count()`), never guess.
* Verify correctness after salting: same row counts and same per-key aggregates (like the check cell above).
* Set `N_SALTS` roughly to the number of usable cores — more salts = more parallelism, but more small-side rows and a bigger shuffle.
* Prefer deterministic salts (`pmod(hash(...), N)`) over `F.rand()` — see the pitfall cell.